# Predicting loan approvals on synthetic data

**Abdulrahman Saad Asiri · Tuwaiq Data Science and AI Bootcamp**

This excerpt adapts the logistic regression baseline from my loan classification submission. I dropped `points` because its correlation with `loan_approved` was 0.82 and it might have encoded the approval decision. I made that choice after inspecting the full dataset, so the held-out result is an internal check. The original exercise retained a synthetic applicant name as a feature. The results apply only to this simulated dataset.

**To reproduce the matrix**, place `loan_approval.csv` beside this notebook and run all cells. The publisher describes the 2,000 applications as synthetic and licenses the dataset under MIT. Tested with Python 3.13.15, pandas 3.0.0, NumPy 2.2.6, and scikit-learn 1.8.0. [Anish Dev Edward, Loan Approval Dataset, version 1](https://www.kaggle.com/datasets/anishdevedward/loan-approval-dataset/versions/1)

In [1]:
from hashlib import sha256
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

csv_path = Path("loan_approval.csv")
expected_hash = "384be11c0f0a3fcb8afb1162797778241f19484f89da3e7a3573d97330340c34"
if sha256(csv_path.read_bytes()).hexdigest() != expected_hash:
    raise ValueError("Use the original loan_approval.csv from dataset version 1.")

data = pd.read_csv(csv_path).drop_duplicates()
features = data.drop(columns=["points", "loan_approved"])
approved = data["loan_approved"]
X_train, X_test, y_train, y_test = train_test_split(
    features, approved, test_size=0.2, stratify=approved, random_state=42
)

In [2]:
numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("numeric", numeric, ["income", "credit_score", "loan_amount", "years_employed"]),
    ("categorical", categorical, ["name", "city"]),
])
baseline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])
predicted = baseline.fit(X_train, y_train).predict(X_test)

## Held-out confusion matrix

The model trained on 1,600 applications. Rows show the recorded outcome for the other 400 applications, and columns show the model’s prediction. The off-diagonal cells count the two kinds of error.

In [3]:
counts = confusion_matrix(y_test, predicted, labels=[False, True])
matrix = pd.DataFrame(
    counts,
    index=["Actually rejected", "Actually approved"],
    columns=["Predicted rejected", "Predicted approved"],
)
print(matrix.to_string())

                   Predicted rejected  Predicted approved
Actually rejected                 213                  11
Actually approved                  14                 162
